In [21]:
import numpy as np 
from uncertainties import ufloat
import pandas as pd 
import matplotlib.pyplot as plt

In [22]:
#plot settings
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "text.latex.preamble": r"\usepackage{amsmath}"
})
plt.rcParams["font.size"] = 15

In [24]:
mcmc_fit_data = pd.read_csv("mcmc_fit_adjusted.csv", index_col = 0)
bell_data = pd.read_csv("bell_final_data.csv")

In [25]:
mcmc_fit_data

,Median Value,Uncertainty,2-sigma Uncertainty,Uncertainty/Value
Phase,-1.478390,0.398976,0.797952,-0.539744
Amplitude,674.381472,4.688559,9.377117,0.013905
Baseline,812.837407,3.809612,7.619224,0.009374


In [26]:
center_line = ufloat(mcmc_fit_data["Median Value"].to_numpy()[2], mcmc_fit_data["Uncertainty"].to_numpy()[2])
amplitude = ufloat(mcmc_fit_data["Median Value"].to_numpy()[1], mcmc_fit_data["Uncertainty"].to_numpy()[1])

background = center_line - amplitude
background

138.45593506641376+/-6.041169048754545

In [27]:
raw_counts = bell_data["N_COIN"].to_numpy()
raw_rate = np.array([ufloat(i, np.sqrt(i)) for i in raw_counts]) / bell_data["TIME"].to_numpy()
raw_rate -= background

#include this if you want to inflate the error bound
chi_sq_uncertainty_multiplier = np.sqrt(6.960199927344692)
for i in range(len(raw_rate)):
    raw_rate[i] = ufloat(raw_rate[i].nominal_value, raw_rate[i].std_dev * chi_sq_uncertainty_multiplier)

raw_rate

array([993.7449304987458+/-32.29162753579611,
       1081.066051657447+/-33.255348563417684,
       285.6385517052582+/-23.43484448887516,
       237.43993007906965+/-22.707833258617917,
       243.23719445725482+/-22.796464621044198,
       198.5406949672859+/-22.10365531604678,
       1066.189697342517+/-33.063626381943926,
       1109.3514361733094+/-33.526937028677416,
       1133.0235510172713+/-33.760104515220114,
       1130.7691806517587+/-33.72493839611425,
       174.2396871948746+/-21.71766308024104,
       116.80691014259969+/-20.80675571727953,
       1166.8930271530805+/-34.13478248628472,
       1091.4776485200725+/-33.31710778473572,
       188.54014098067364+/-21.94562843121747,
       176.02143397307907+/-21.750707869369194], dtype=object)

In [28]:
def E(rates, start):
    return (rates[start] + rates[start + 1] - rates[start + 2] - rates[start + 3])/np.sum(rates[start:start+4])

In [29]:
bell_data

,TRIAL_NO,ORIENT,ARGUMENT_1,ARGUMENT_2,ANGLE_A,ANGLE_B,N_A,N_B,N_COIN,TIME,N_ACTUAL
0,1,N(VV),A,B,-45,-22.5,418879,275446,11312,9.99116,11124.8
1,2,N(HH),A+90,B+90,45,67.5,420169,270848,12151,9.96374,11965.9
2,3,N(VH),A,B+90,-45,67.5,420894,273547,4241,10.00013,4054.4
3,4,N(HV),A+90,B,45,-22.5,421810,275305,3759,10.00011,3570.8
4,5,N(VV),A,B_PRIME,-45,22.5,419836,293497,3817,10.00018,3617.3
5,6,N(HH),A + 90,B_PRIME + 90,45,112.5,420581,254665,3370,10.00010,3196.4
6,7,N(VH),A,B_PRIME + 90,-45,112.5,419636,254455,12036,9.99132,11862.8
7,8,N(HV),A + 90,B_PRIME,45,22.5,420469,292662,12456,9.98231,12256.2
8,9,N(VV),A_PRIME,B,0,-22.5,441672,273289,12704,9.99151,12508.2
9,10,N(HH),A_PRIME + 90,B + 90,90,67.5,401713,276152,12693,10.00059,12513.2


In [30]:
E1 = E(raw_rate,0)
E2 = E(raw_rate,4)
E3 = E(raw_rate,8)
E4 = E(raw_rate,12)
S = E1 - E2 + E3 + E4

In [31]:
for i, e in enumerate([E1, E2, E3, E4]):
    print(f"E{i+1}:", e)

E1: 0.597+/-0.021
E2: -0.662+/-0.021
E3: 0.772+/-0.021
E4: 0.722+/-0.021


In [32]:
print("S:", S)

S: 2.75+/-0.04
